# NYC Mobility - Analytics Validation

## Validation scope

This notebook validates the three business-ready Analytics views produced from the Gold star schema:

- `analytics_taxi_demand`
- `analytics_weather_behavior`
- `analytics_area_mobility_patterns`

The checks confirm completeness, uniqueness, validity, and consistency. Each result is calculated directly from the current Analytics views and the accepted in-scope Gold fact population. The notebook records the detailed evidence, creates a consolidated PASS/FAIL summary, and enforces the final quality gate.

## Verified conditions

The Analytics layer is aligned when all of these conditions return `PASS`:

1. Each business-ready view contains records.
2. Taxi demand has one row per pickup date, clock hour, and pickup Taxi Zone.
3. Taxi demand uses valid clock hours from 0 through 23.
4. Area mobility has one row per Taxi Zone.
5. Taxi demand and weather behavior trip volumes reconcile with the in-scope Gold fact population.
6. Area pickup and drop-off volumes each reconcile with the same in-scope Gold fact population.

The common fact scope uses `dq_out_of_range_datetime = FALSE`, matching the Analytics creation queries.

## Create detailed Analytics validation results

This step creates `analytics_validation_results`. Every row represents one governed quality check with:

- the check name;
- the data quality attribute;
- the expected condition;
- the value calculated from current data; and
- the resulting `PASS` or `FAIL` status.

### Data quality attributes

- **Completeness** confirms that each Analytics view contains usable records.
- **Uniqueness** confirms that each declared business grain appears once.
- **Validity** confirms that hour values follow the 0–23 clock convention.
- **Consistency** confirms that Analytics totals reconcile with the Gold fact scope.

In [ ]:
CREATE OR REPLACE VIEW `ftw-week-08`.`03_gold`.analytics_validation_results AS
WITH fact_scope AS (
  SELECT COUNT(*) AS in_scope_fact_rows
  FROM `ftw-week-08`.`03_gold`.fact_green_taxi_trip
  WHERE dq_out_of_range_datetime = FALSE
),
taxi_demand_duplicate_grain AS (
  SELECT COUNT(*) AS duplicate_groups
  FROM (
    SELECT
      pickup_date,
      pickup_hour_24,
      pickup_taxi_zone_key,
      COUNT(*) AS group_rows
    FROM `ftw-week-08`.`03_gold`.analytics_taxi_demand
    GROUP BY pickup_date, pickup_hour_24, pickup_taxi_zone_key
    HAVING COUNT(*) > 1
  )
),
area_duplicate_grain AS (
  SELECT COUNT(*) AS duplicate_groups
  FROM (
    SELECT taxi_zone_key, COUNT(*) AS group_rows
    FROM `ftw-week-08`.`03_gold`.analytics_area_mobility_patterns
    GROUP BY taxi_zone_key
    HAVING COUNT(*) > 1
  )
)
SELECT
  'Taxi demand view contains rows' AS check_name,
  'COMPLETENESS' AS quality_attribute,
  '> 0' AS expected_value,
  CAST(COUNT(*) AS STRING) AS actual_value,
  CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END AS status
FROM `ftw-week-08`.`03_gold`.analytics_taxi_demand

UNION ALL

SELECT
  'Taxi demand declared grain is unique',
  'UNIQUENESS',
  '0 duplicate groups',
  CONCAT(CAST(duplicate_groups AS STRING), ' duplicate groups'),
  CASE WHEN duplicate_groups = 0 THEN 'PASS' ELSE 'FAIL' END
FROM taxi_demand_duplicate_grain

UNION ALL

SELECT
  'Taxi demand uses clock hours 0-23',
  'VALIDITY',
  '0 invalid hours',
  CONCAT(CAST(COUNT_IF(pickup_hour_24 NOT BETWEEN 0 AND 23) AS STRING), ' invalid hours'),
  CASE
    WHEN COUNT_IF(pickup_hour_24 NOT BETWEEN 0 AND 23) = 0 THEN 'PASS'
    ELSE 'FAIL'
  END
FROM `ftw-week-08`.`03_gold`.analytics_taxi_demand

UNION ALL

SELECT
  'Taxi demand trip volume reconciles with Gold',
  'CONSISTENCY',
  CAST(f.in_scope_fact_rows AS STRING),
  CAST(COALESCE(SUM(a.trip_volume), 0) AS STRING),
  CASE
    WHEN COALESCE(SUM(a.trip_volume), 0) = f.in_scope_fact_rows THEN 'PASS'
    ELSE 'FAIL'
  END
FROM `ftw-week-08`.`03_gold`.analytics_taxi_demand AS a
CROSS JOIN fact_scope AS f
GROUP BY f.in_scope_fact_rows

UNION ALL

SELECT
  'Weather behavior view contains rows',
  'COMPLETENESS',
  '> 0',
  CAST(COUNT(*) AS STRING),
  CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END
FROM `ftw-week-08`.`03_gold`.analytics_weather_behavior

UNION ALL

SELECT
  'Weather behavior trip volume reconciles with Gold',
  'CONSISTENCY',
  CAST(f.in_scope_fact_rows AS STRING),
  CAST(COALESCE(SUM(a.trip_volume), 0) AS STRING),
  CASE
    WHEN COALESCE(SUM(a.trip_volume), 0) = f.in_scope_fact_rows THEN 'PASS'
    ELSE 'FAIL'
  END
FROM `ftw-week-08`.`03_gold`.analytics_weather_behavior AS a
CROSS JOIN fact_scope AS f
GROUP BY f.in_scope_fact_rows

UNION ALL

SELECT
  'Area mobility view contains rows',
  'COMPLETENESS',
  '> 0',
  CAST(COUNT(*) AS STRING),
  CASE WHEN COUNT(*) > 0 THEN 'PASS' ELSE 'FAIL' END
FROM `ftw-week-08`.`03_gold`.analytics_area_mobility_patterns

UNION ALL

SELECT
  'Area mobility declared grain is unique',
  'UNIQUENESS',
  '0 duplicate groups',
  CONCAT(CAST(duplicate_groups AS STRING), ' duplicate groups'),
  CASE WHEN duplicate_groups = 0 THEN 'PASS' ELSE 'FAIL' END
FROM area_duplicate_grain

UNION ALL

SELECT
  'Area pickup volume reconciles with Gold',
  'CONSISTENCY',
  CAST(f.in_scope_fact_rows AS STRING),
  CAST(COALESCE(SUM(a.pickup_trip_volume), 0) AS STRING),
  CASE
    WHEN COALESCE(SUM(a.pickup_trip_volume), 0) = f.in_scope_fact_rows THEN 'PASS'
    ELSE 'FAIL'
  END
FROM `ftw-week-08`.`03_gold`.analytics_area_mobility_patterns AS a
CROSS JOIN fact_scope AS f
GROUP BY f.in_scope_fact_rows

UNION ALL

SELECT
  'Area drop-off volume reconciles with Gold',
  'CONSISTENCY',
  CAST(f.in_scope_fact_rows AS STRING),
  CAST(COALESCE(SUM(a.dropoff_trip_volume), 0) AS STRING),
  CASE
    WHEN COALESCE(SUM(a.dropoff_trip_volume), 0) = f.in_scope_fact_rows THEN 'PASS'
    ELSE 'FAIL'
  END
FROM `ftw-week-08`.`03_gold`.analytics_area_mobility_patterns AS a
CROSS JOIN fact_scope AS f
GROUP BY f.in_scope_fact_rows;


## Create the consolidated validation summary

This step creates `analytics_validation_summary`. It counts all checks, separates passed and failed checks, records the validation timestamp, and sets the overall status:

- `PASS` when every detailed check passes;
- `FAIL` when one or more detailed checks fail.

The summary provides one governed result for the Analytics validation task and the end-to-end quality gate.

In [ ]:
CREATE OR REPLACE VIEW `ftw-week-08`.`03_gold`.analytics_validation_summary AS
SELECT
  CURRENT_TIMESTAMP() AS checked_at,
  COUNT(*) AS total_checks,
  COUNT_IF(status = 'PASS') AS passed_checks,
  COUNT_IF(status = 'FAIL') AS failed_checks,
  CASE WHEN COUNT_IF(status = 'FAIL') = 0 THEN 'PASS' ELSE 'FAIL' END AS overall_status
FROM `ftw-week-08`.`03_gold`.analytics_validation_results;


## Detailed validation evidence

The result set below displays every Analytics check in a consistent order. Review `expected_value`, `actual_value`, and `status` together to identify the exact view and quality attribute represented by each result.

In [ ]:
SELECT *
FROM `ftw-week-08`.`03_gold`.analytics_validation_results
ORDER BY quality_attribute, check_name;


## Consolidated Analytics status

The summary below presents the current validation timestamp, total checks, passed checks, failed checks, and overall status. These values are calculated from `analytics_validation_results`, so the summary remains aligned with the detailed evidence.

In [ ]:
SELECT *
FROM `ftw-week-08`.`03_gold`.analytics_validation_summary;


## Enforce the Analytics quality gate

The final assertion converts the consolidated status into an executable quality gate. A zero failed-check count completes the validation successfully. Any failed check stops the task and reports the number of failed conditions for investigation.

In [ ]:
SELECT ASSERT_TRUE(
  failed_checks = 0,
  CONCAT('Analytics validation failed: ', CAST(failed_checks AS STRING), ' check(s) failed')
)
FROM `ftw-week-08`.`03_gold`.analytics_validation_summary;


## Analytics validation complete

The notebook creates detailed and consolidated validation views, verifies the declared Analytics grains and clock-hour convention, reconciles business-ready trip volumes with the governed Gold scope, and enforces the final PASS/FAIL gate.

All numerical evidence is calculated by the SQL cells from the current Databricks tables. The documentation describes the verified conditions without inserting assumed results.